In [3]:
#Imports

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (

    train_test_split,

    cross_val_score

)

from sklearn.metrics import (

    accuracy_score

)

from xgboost import (

    XGBClassifier

)

In [4]:
#Configurações do MLflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("XGboost")

mlflow.sklearn.autolog()

2026/05/24 18:03:41 INFO mlflow.tracking.fluent: Experiment with name 'XGboost' does not exist. Creating a new experiment.


In [5]:
#CArregar o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [6]:
#Preparar os dados

df["YearsCodePro_Num"]=(

    df["YearsCodePro"]

)

df["YearsCodePro_Num"]=(
    df["YearsCodePro_Num"]

    .replace({

        "Less than 1 year":0,

        "More than 50 years":51,

        "Sem dado":0

    })
)

df["YearsCodePro_Num"]=pd.to_numeric(

    df["YearsCodePro_Num"]

)

features=[

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code"

]

target="JobSat_Class"

X=df[features]

y=df[target]

y=y.replace({

    "Baixo":0,

    "Medio":1,

    "Alto":2

})

C:\Users\luisr\AppData\Local\Temp\ipykernel_21264\1816381890.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y=y.replace({


In [7]:
#Split dos dados

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

In [8]:
#Função do XGBoost

def run_xgb_experiment(

    run_name,

    estimators,

    lr,

    depth

):

    with mlflow.start_run(

        run_name=run_name

    ):

        model=XGBClassifier(

            n_estimators=
            estimators,

            learning_rate=
            lr,

            max_depth=
            depth,

            random_state=42,

            eval_metric=
            "mlogloss"

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred=model.predict(

            X_test

        )

        accuracy=accuracy_score(

            y_test,

            y_pred

        )

        cv=cross_val_score(

            model,

            X_train,

            y_train,

            cv=5

        )

        mlflow.log_metric(

            "accuracy",

            accuracy

        )

        mlflow.log_metric(

            "cv_mean",

            cv.mean()

        )

        print(run_name)

        print(
            accuracy
        )

In [9]:
#Experiencia 0 - Baseline

run_xgb_experiment(

    "Experiment_0_Baseline",
    100,
    0.1,
    3
)

2026/05/24 18:03:55 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Experiment_0_Baseline
0.7495466634429401


In [10]:
#Experiencia 1 -

run_xgb_experiment(

    "Experiment_1_200",
    200,
    0.1,
    3
)

Experiment_1_200
0.7504231141199227


In [11]:
#Experiencia 2 

run_xgb_experiment(
    "Experiment_2_lr005",
    100,
    0.05,
    3
)

Experiment_2_lr005
0.7482168762088974


In [12]:
#Experiencia 3

run_xgb_experiment(

    "Experiment_3_lr02",
    100,
    0.2,
    3
)

Experiment_3_lr02
0.7505137814313346


In [13]:
#Experiencia 4

run_xgb_experiment(

    "Experiment_4_depth5",
    100,
    0.1,
    5
)

Experiment_4_depth5
0.7534755802707931


In [14]:
#Experiencia 5

features_extra=[
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5"
]

X=df[features_extra]

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

run_xgb_experiment(
    "Experiment_5_Features",
    100,
    0.1,
    3
)

Experiment_5_Features
0.757948500967118
